# Heat Radiation Lab (Notebook)
Author: Harpreet Singh · MIT License

Sections: physics, data cleaning, AI ethics, analysis, 3D/graphs/charts, ML, DL, computer vision, full-stack demo.
Run from the repo folder (flat layout).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
from radiation import net_flux, planck, wien_peak, view_factor_parallel_discs
print('Flux 800K, e=0.8:', net_flux(0.8, 800, 300), 'W/m2')

## 1. Data cleaning (pandas)

In [ ]:
raw = pd.read_csv('emissivity_raw.csv')
print(raw.shape); display(raw.isna().sum()); print('duplicates:', raw.duplicated().sum())
print('bad temps:', (raw.temp_K<=0).sum(), ' bad emissivity:', (~raw.emissivity.between(0,1) & raw.emissivity.notna()).sum())

In [ ]:
df = raw.drop_duplicates().copy()
log = {'dupes_removed': len(raw)-len(df)}
df['material'] = df.material.str.strip().str.capitalize()
bad_t = df.temp_K <= 0; df.loc[bad_t,'temp_K'] = np.nan; log['bad_temp'] = int(bad_t.sum())
bad_e = ~df.emissivity.between(0,1) & df.emissivity.notna(); df.loc[bad_e,'emissivity'] = np.nan; log['bad_emissivity'] = int(bad_e.sum())
log['emissivity_imputed'] = int(df.emissivity.isna().sum())
df['emissivity'] = df.groupby('material').emissivity.transform(lambda s: s.fillna(s.median()))
n0 = len(df); df = df.dropna(subset=['temp_K','emissivity']).reset_index(drop=True); log['rows_dropped'] = n0-len(df)
df['flux_Wm2'] = net_flux(df.emissivity, df.temp_K, df.surr_K); df['power_W'] = df.flux_Wm2*df.area_m2
print(log); df.describe().round(2)

## 2. AI ethics checks
See `AI_ETHICS.md`. Below: coverage, imputation share, and an extrapolation guard.

In [ ]:
print('Synthetic data only - not real measurements.')
cover = df.material.value_counts(); display(cover)
print('Imputed share: %.1f%%' % (100*log['emissivity_imputed']/len(raw)))
TMIN, TMAX = df.temp_K.min(), df.temp_K.max()
def safe_predict_range(T):
    return 'OK' if TMIN <= T <= TMAX else f'WARNING: {T} K outside training range {TMIN:.0f}-{TMAX:.0f} K'
print(safe_predict_range(2500)); print('Not for safety-critical use without expert review.')

## 3. Data analyst: comparison table

In [ ]:
table = df.groupby('material').agg(n=('power_W','size'), emissivity=('emissivity','mean'), mean_flux=('flux_Wm2','mean'), mean_power=('power_W','mean')).round(2).sort_values('mean_flux', ascending=False)
table.to_csv('comparison_table.csv'); table

## 4. 3D diagram, graphs and charts

In [ ]:
T = np.linspace(300,1500,60); E = np.linspace(.05,1,60); TT,EE = np.meshgrid(T,E)
fig = plt.figure(figsize=(8,6)); ax = fig.add_subplot(111, projection='3d')
s = ax.plot_surface(TT,EE,net_flux(EE,TT,300)/1e3,cmap='inferno'); ax.set_xlabel('T (K)'); ax.set_ylabel('Emissivity'); ax.set_zlabel('kW/m2'); fig.colorbar(s,shrink=.6); plt.show()

In [ ]:
wl = np.linspace(.1e-6,20e-6,600); plt.figure(figsize=(8,5))
for Tk in (500,1000,2000,3000,5778):
    plt.plot(wl*1e6, planck(wl,Tk), label=f'{Tk} K'); plt.scatter(wien_peak(Tk)*1e6, planck(wien_peak(Tk),Tk), c='k', s=12)
plt.yscale('log'); plt.ylim(1e6,None); plt.xlabel('um'); plt.legend(); plt.title('Planck + Wien'); plt.show()

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4))
table.mean_flux.sort_values().plot.barh(ax=ax[0], color='darkorange', title='Mean flux by material')
df.groupby('material').power_W.sum().plot.pie(ax=ax[1], textprops={'fontsize':6}, ylabel='', title='Power share')
sc = ax[2].scatter(df.temp_K, df.flux_Wm2/1e3, c=df.emissivity, s=10, cmap='viridis'); fig.colorbar(sc, ax=ax[2]); ax[2].set_title('Flux vs T'); plt.tight_layout(); plt.show()

## 5. Machine learning

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
X, y = df[['temp_K','emissivity','surr_K']], df.flux_Wm2
Xt,Xv,yt,yv = train_test_split(X,y,test_size=.25,random_state=0)
rf = RandomForestRegressor(200, random_state=0).fit(Xt,yt); p = rf.predict(Xv)
print('R2 %.4f  MAE %.1f' % (r2_score(yv,p), mean_absolute_error(yv,p)))
pd.Series(rf.feature_importances_, X.columns).plot.bar(title='Feature importance'); plt.show()

## 6. Deep learning (numpy MLP)

In [ ]:
rng = np.random.default_rng(0)
Xd = df[['temp_K','emissivity']].values/np.array([1500.,1.]); yd = (df.flux_Wm2.values/1e5)[:,None]
n = int(.8*len(Xd)); Xt,Xv,yt,yv = Xd[:n],Xd[n:],yd[:n],yd[n:]
W1=rng.normal(0,.3,(2,32)); b1=np.zeros(32); W2=rng.normal(0,.3,(32,32)); b2=np.zeros(32); W3=rng.normal(0,.3,(32,1)); b3=np.zeros(1)
def fwd(x):
    h1=np.tanh(x@W1+b1); h2=np.tanh(h1@W2+b2); return h1,h2,h2@W3+b3
losses=[]
for ep in range(6000):
    h1,h2,o = fwd(Xt); g = 2*(o-yt)/len(Xt); losses.append(float(((o-yt)**2).mean()))
    gW3=h2.T@g; g2=(g@W3.T)*(1-h2**2); gW2=h1.T@g2; g1=(g2@W2.T)*(1-h1**2); gW1=Xt.T@g1
    W3-=.01*gW3; b3-=.01*g.sum(0); W2-=.01*gW2; b2-=.01*g2.sum(0); W1-=.01*gW1; b1-=.01*g1.sum(0)
pv = fwd(Xv)[2]; print('MLP R2 = %.4f' % (1-((pv-yv)**2).sum()/((yv-yv.mean())**2).sum()))
plt.plot(losses); plt.yscale('log'); plt.title('Training loss'); plt.show()

## 7. Computer vision: thermal hot spots

In [ ]:
from scipy import ndimage
r = np.random.default_rng(1); yy,xx = np.mgrid[0:200,0:200]; img = 300 + r.normal(0,1.5,(200,200))
for cx,cy,s_,a in [(50,60,10,220),(140,120,14,330),(150,40,7,150)]: img += a*np.exp(-((xx-cx)**2+(yy-cy)**2)/(2*s_**2))
img = ndimage.gaussian_filter(img,1); mask = img > img.mean()+3*img.std(); lab,k = ndimage.label(mask); print('Hot spots:', k)
fig,ax = plt.subplots(1,3,figsize=(13,4))
for a_,im_,t_,c_ in zip(ax,[img,mask,net_flux(.9,img,300)/1e3],['Thermal (K)','Mask','Flux kW/m2'],['inferno','gray','magma']): a_.imshow(im_,cmap=c_); a_.set_title(t_); a_.axis('off')
plt.show()

## 8. Full-stack development
Backend `server.py` (stdlib HTTP + JSON API) and frontend `index.html`. Here we start the server in a thread and call the API from the notebook.

In [ ]:
import threading, json, urllib.request, time
from http.server import ThreadingHTTPServer
from server import Handler
srv = ThreadingHTTPServer(('127.0.0.1', 8765), Handler); threading.Thread(target=srv.serve_forever, daemon=True).start(); time.sleep(.5)
get = lambda p: json.load(urllib.request.urlopen('http://127.0.0.1:8765'+p))
print(get('/api/flux?e=0.8&T=800&Ts=300'))
try: get('/api/flux?e=5&T=800')
except Exception as ex: print('validation works:', ex)
print(get('/api/materials')[:2]); srv.shutdown()

## 9. Other languages
Same physics in `radiation.go/.rs/.c/.cpp/.js/.ts/.sh`, `Radiation.java`, `analysis.R`.

## 10. New physics: surface exchange, radiation shields, cooling

In [ ]:
from radiation import exchange_two_surfaces, shield_flux, cooling_curve
print('Exchange Q (W):', exchange_two_surfaces(.8,.8,1,1,600,300))
for n in (0,1,2,5): print(n, 'shields ->', round(float(shield_flux(.8,.8,.1,600,300,n)),1), 'W/m2')
t,Tc = cooling_curve(1000,300,.9,1,10,500,3000,10)
plt.plot(t/60,Tc); plt.xlabel('min'); plt.ylabel('K'); plt.title('Radiative cooling'); plt.show()

## 11. Diagrams (architecture, flow, 3D, spectrum, shields, view factor, correlation, radar)

In [ ]:
import diagrams
from IPython.display import Image, display
diagrams.architecture(); diagrams.cleaning_flow(); diagrams.sphere_3d(); diagrams.cooling_3d(); diagrams.exchange_3d()
diagrams.em_spectrum(); diagrams.shield_network(); diagrams.view_factor_map(); diagrams.corr_heatmap(df); diagrams.radar(df)
for f in ['architecture','cleaning_flow','3d_sphere','3d_cooling','3d_exchange','em_spectrum','shield_network','view_factor','correlation','radar']:
    display(Image(f'diagram_{f}.png'))

## 12. Extra physics, hybrid ML and uncertainty

In [ ]:
import radiation_extra as x
print('Tungsten-like total emissivity 1000K/3000K:', round(x.total_emissivity(1000),3), round(x.total_emissivity(3000),3))
print('Stagnation T selective, C=10:', round(x.stagnation_temperature(.95,.1,1000,10)), 'K')
print('Satellite T:', round(x.satellite_temperature(.3,.8,1,4)), 'K')
print('Parallel square view factor:', round(float(x.view_factor_parallel_rects(1,1,1)),4))
import diagrams_extra as de
de.emissivity(); de.collector(); de.satellite_3d(); de.rect_view_factors()
for f in ['emissivity_spectral','solar_collector','3d_satellite','view_factor_rects']: display(Image(f'diagram_{f}.png'))

In [ ]:
%run hybrid_ml.py

## 13. Thermal helpers, PDF report and language benchmark

In [ ]:
import thermal, report, io
G = thermal.to_temperature(img, 290, 420)
mask2, lab2, spots = thermal.detect_hotspots(G)
print(pd.DataFrame(spots))
buf = io.BytesIO(); report.build_report(df, buf, G, spots); open('heat_radiation_report.pdf','wb').write(buf.getvalue()); print('PDF bytes:', len(buf.getvalue()))

In [ ]:
%run compare_languages.py 1000000
display(Image('diagram_benchmark.png'))